# 02 — Watermarks y datos tardíos

En el mundo real, los eventos **no llegan en orden**. Un evento con
`event_time=10:00:00` puede llegar a las 10:00:30. Spark necesita saber
cuánto esperar antes de cerrar una ventana. Eso es el **watermark**.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("03_02_watermarks")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## El problema: sin watermark, Spark espera para siempre

In [ ]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

sin_watermark = (
    stream
    .groupBy(F.window(F.col("timestamp"), "1 minute"))
    .count()
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("sin_watermark")
    .start()
)
print("Query sin watermark arrancada")

## Con watermark: ventana se cierra después de 10 min de inactividad

In [ ]:
con_watermark = (
    stream
    .withWatermark("timestamp", "10 minutes")
    .groupBy(F.window(F.col("timestamp"), "5 minutes"))
    .count()
    .writeStream
    .outputMode("update")
    .format("memory")
    .queryName("con_watermark")
    .start()
)
print("Query con watermark arrancada")

In [ ]:
import time
time.sleep(15)
print("--- con_watermark ---")
spark.sql("SELECT * FROM con_watermark ORDER BY window.start").show(truncate=False)
sin_watermark.stop()
con_watermark.stop()

## Cuándo usar cada `outputMode`

| Modo | Qué escribe | Necesita watermark? | Cuándo |
|------|-------------|---------------------|--------|
| `append` | Solo filas nuevas que ya no pueden cambiar | **Sí** (con aggregations) | Cuando solo necesitás el cierre de la ventana |
| `update` | Filas que cambiaron en el último batch | Sí | Default moderno para aggregates |
| `complete` | Toda la tabla de salida reescrita | No | State chico (counts globales) |

**Regla práctica**: si tu agregación tiene ventana temporal, **siempre
poné watermark**. Sin él, el state crece para siempre.

## Trade-off: ¿cuánto watermark?

- **Muy poco (1 min)**: bajo retraso de cierre, pero perdés eventos
  que llegan tarde.
- **Mucho (1 hora)**: ves todos los eventos, pero la ventana tarda en
  cerrarse y consumís más memoria.

**Regla**: watermark = p99 del retraso de tus datos. Si el 99% de los
eventos llega en menos de 30s, watermark = `1 minute` es suficiente.

## ✅ Checkpoint

- [x] Sé que sin watermark Spark espera para siempre.
- [x] Sé poner `withWatermark` y elegir `outputMode` correcto.
- [x] Sé balancear latencia vs completitud.

**Siguiente**: `03_files_source_delta_sink.ipynb` — leer de un
directorio y escribir a Delta Lake.